# 7. Biosensor information experiments

## Goal

Which reporter channels reduce decision-relevant uncertainty or make edit effects identifiable, and which apparent gains disappear under leakage and negative controls?

This notebook is a readable research record. It follows the actual hand-offs in order and loads the saved evidence by default; it does **not** hide the experiment behind a one-cell runner.


## Pipeline at a glance

```text
goal → declared generator → observable/lockbox split → model setup & training
     → candidate or condition screen → matched comparison → interpretation
```

Each section below corresponds to one of these hand-offs.


In [ ]:
# Run this notebook from the repository root.
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

REGENERATE = False  # Cached artifacts are the default; no expensive solve runs implicitly.

def artifact(relative_path: str) -> Path:
    """Fail with a useful message rather than silently replacing evidence."""
    path = ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(f"Missing cached artifact: {path}")
    return path

def show(frame, n=8):
    # `print` keeps this notebook usable in a plain Python kernel as well as Jupyter.
    print(frame.head(n).to_string(index=False))
    print(f"{len(frame):,} rows × {len(frame.columns):,} columns")


## 1. Experimental contract

The experiment has a declared observation boundary. “Observable” means the learner may use it; “lockbox” means it may be generated and audited but must not be used as a deployable feature.


In [ ]:
from yeast_validation import run_rxncon_reporter_supervision_diagnostic as diagnostic
from yeast_validation import run_rxncon_edit_identifiability_live_biosensor as ident

print("diagnostic module:", diagnostic.__name__)
print("edit-identifiability module:", ident.__name__)
print("contract: reporters observable; regulatory state and GSM controls lockboxed")


## 2. Data generator

This phase uses rxncon/GSM cultures with reporter observations separated from hidden regulatory histories and GSM-interface controls.

The next cell exposes the generator’s first concrete hand-off. It is deliberately small/inspection-only where generating the full campaign is expensive.


In [ ]:
# Read the explicit reporter experiment record before interpreting an aggregate metric.
metrics = pd.read_csv(artifact("data/rxncon_reporter_supervision_diagnostic/reporter_diagnostic_summary_table.csv"))
show(metrics)


## 3. Model setup and training contract

Assimilation and edit-inference models receive only the selected reporter/history interface. The notebook explicitly separates reporter-supervision and live-biosensor-identifiability analyses.

Training is not automatically started in this notebook. The cached training/evaluation artifacts below are the evidence record; regeneration must be an intentional, parameterized action.


In [ ]:
# Make the experiment hand-off inspectable before looking at aggregate metrics.
for name, relative_path in [('metrics', 'data/rxncon_reporter_supervision_diagnostic/reporter_diagnostic_summary_table.csv')]:
    path = artifact(relative_path)
    print(f"{name}: {path.relative_to(ROOT)}")


## 4. Screening / selection stage

Screen reporter sets and edit hypotheses under shared cultures, with shuffled/noisy/missing/irrelevant controls retained as first-class results.

The screen is intentionally shown separately from final verification, so a virtual score cannot be mistaken for an exact outcome.


In [ ]:
# Load the primary evidence table and inspect its schema before aggregation.
metrics = pd.read_csv(artifact('data/rxncon_reporter_supervision_diagnostic/reporter_diagnostic_summary_table.csv'))
show(metrics)


## 5. Matched comparison

Compare uncertainty, prediction, or edit-identification metrics at the same observation budget and culture split.


In [ ]:
# Aggregate only over fields that exist in this version of the cached record.
comparison = metrics.groupby(['condition_family', 'reporter_mode', 'reporter_set'], dropna=False).mean(numeric_only=True)
show(comparison.reset_index() if hasattr(comparison, "reset_index") else comparison)


## 6. Analysis view

The plot is intentionally generic: it exposes every numeric evidence column so the reader can select the metric relevant to the claim, rather than hard-coding an attractive subset.


In [ ]:
numeric = metrics.select_dtypes("number")
if numeric.shape[1]:
    ax = numeric.plot(kind="box", rot=45, figsize=(11, 4), title="Cached evidence: numeric metric distribution")
    ax.set_ylabel("recorded metric value")
    plt.tight_layout()
else:
    print("This artifact has no numeric columns to plot.")


## 7. Interpretation, scope, and next hand-off

A channel matters only when it changes a deployable inference or decision; correlation with product alone is not enough.

### Reproduction boundary

The cells above reveal the inputs and artifacts without launching an expensive campaign. To regenerate, use the explicit command below only after reviewing its declared inputs and output destination.


In [ ]:
if REGENERATE:
    # This guard prevents accidental solver/campaign execution.
    raise RuntimeError('Run the reporter diagnostic and identifiability campaigns with their explicit declared arguments; they are not hidden notebook side effects.')
